# Marker Repo - annotation zebrahub using homology

This example shows the application of the homology functions of the Marker Repo by annotating and comparing already annotated data from Zebrahub with the transferred data from PanglaoDB.

## Loading packages

In [ ]:
import markerrepo.marker_repo as mr
import markerrepo.wrappers as wrap
import markerrepo.annotation as annot
import scanpy as sc

%load_ext autoreload
%autoreload 2

## Settings

Specify path of the cloned repository and the h5ad file which is going to be annotated.

In [ ]:
repo_path = "/mnt/workspace/mkessle/projects/annotate_by_marker_and_features"
h5ad_path = "/mnt/workspace/mkessle/master/refdata/zf_atlas_10dpf_v4_release.h5ad"

Load anndata

In [ ]:
adata = sc.read_h5ad(h5ad_path)

Filter adata to keep clusters of 12 cell types only

In [ ]:
wanted_cell_types = [
    'Muller cell',
    'amacrine cell',
    'central nervous system',
    'cartilage element',
    'liver',
    'kidney cell',
    'macrophage',
    'muscle',
    'neuronal stem cell',
    'intestine',
    'dermis',
    'vasculature'
]

adata_filtered = adata[adata.obs['zebrafish_anatomy_ontology_class'].isin(wanted_cell_types)]

List all possible settings.

In [ ]:
annot.list_possible_settings(repo_path, adata_filtered)

In [ ]:
# taxonomy ID or organism name e.g. "human" or 9606
organism = "zebrafish" 
# the column of the .obs table where the ranked genes groups are stored e.g. "rank_genes_groups"
# if no ranking has been performed yet, enter None
rank_genes_column = None
# the column of the .var table where the gene symbols or ensembl IDs or stored
# enter None if the index column of the .var table are already gene symbols or ensembl IDs 
# which you want to use for your annotation
genes_column = None
# the .obs table column of the clustering you want to annotate e.g. "leiden" or "louvain"
column = "zebrafish_anatomy_ontology_class"
# specify wether your index of the .var tables are ensembl IDs (True) or gene symbols (False)
ensembl = False
# specify the marker lists selection you want to use for the annotation
# the column to search in, None to search all columns, e.g. "source", "Organism name", etc.
col_to_search = "Source"
# search terms: "-" exclude keyword, "+" must contain keyword
# separate keywords with  "," e.g. ["+panglao.se", "+mouse"]
search_terms = ["panglao.se"]

Validate input and load anndata object

In [ ]:
annot.validate_settings(repo_path, adata_filtered, organism, rank_genes_column, genes_column, column, ensembl, col_to_search, search_terms)

## Prepare annotation

### Ranking

Rank genes, if not already done.

In [ ]:
if not rank_genes_column:
    rank_genes_column = f'rank_genes_groups_{column}'
    print(f'Ranking genes groups for clusters using obs column {column}')
    sc.tl.rank_genes_groups(adata_filtered, groupby=f'{column}', use_raw=False, key_added=rank_genes_column)

In [ ]:
sc.pl.rank_genes_groups_matrixplot(adata_filtered, standard_scale='var', n_genes=10, key=rank_genes_column, show=True)

## Create suitable marker list(s)

The paths of the marker lists will be stored in the <b>marker_lists</b> variable. They will work as input for the actual cell type annotation of the next cell. If the index of adata.var contains ensembl IDs, set <b>ensembl=True</b>, otherwise gene symbols are used.

Add transferred marker lists using the whole PanglaoDB as source.

In [ ]:
organism = mr.update_organism(organism, repo_path)
marker_lists = wrap.create_marker_lists(organism, repo_path=repo_path, 
                                        style="score", file_name="zebrahub_transferred", ensembl=ensembl,
                                        col_to_search=col_to_search, search_terms=search_terms, force_homology=True)

Add non-transferred marker lists from mouse and human

In [ ]:
col_to_search = "Source"
search_terms = ["panglao.se"]

marker_list_mouse = wrap.create_marker_lists("mouse", repo_path=repo_path, 
                                        style="score", file_name="zebrahub_no_homology_mouse", ensembl=ensembl,
                                        col_to_search=col_to_search, search_terms=search_terms, force_homology=False)
marker_list_human = wrap.create_marker_lists("human", repo_path=repo_path, 
                                        style="score", file_name="zebrahub_no_homology_human", ensembl=ensembl,
                                        col_to_search=col_to_search, search_terms=search_terms, force_homology=False)

marker_lists.extend([lst[0] for lst in [marker_list_mouse, marker_list_human] if lst])

## Annotate adata using the created marker lists

In [ ]:
for marker_list in marker_lists:
    name = marker_list.split("/")[-1]
    annotation_dir = f"./annotation/{name}"
    
    # Annotate
    annot.annot_ct(adata_filtered, output_path=annotation_dir, db_path=marker_list,
                   cluster_column=f"{column}", rank_genes_column=rank_genes_column, 
                   ct_column=f"cell_types_{name}")
    
    # Plot annotation
    sc.pl.umap(adata_filtered, color=[f'cell_types_{name}', f'{column}'], wspace=0.5)

    # Show scores and alternate cell types of eacht cluster
    print(f"Tables of cell type annotation with clustering {column}")
    annot.show_tables(annotation_dir=annotation_dir, n=5, clustering_column=column, show_diff=True)

### Compare cell type annotations from Zebrahub and MarkerRepo

In [ ]:
compare_df = annot.compare_cell_types(adata_filtered, column, marker_lists)

In [ ]:
unique_counts = compare_df.nunique()
display(unique_counts)

In [ ]:
compare_df.rename(columns={compare_df.columns[0]: "HG mouse",
                           compare_df.columns[1]: "HG human",
                           compare_df.columns[2]: "BM mouse",
                           compare_df.columns[3]: "BM human",
                           compare_df.columns[4]: "NT mouse",
                           compare_df.columns[5]: "NT human"}, inplace=True)
compare_df.index.name = "Zebrahub"
unique_counts = compare_df.nunique()
display(unique_counts)
display(compare_df)

### Save UMAP plot from Zebrahub with the analysed cell type clusters only

In [ ]:
adata.obs['filtered_cell_types'] = [cell_type if cell_type in wanted_cell_types else None for cell_type in adata.obs['zebrafish_anatomy_ontology_class']]
sc.pl.umap(adata, color=['filtered_cell_types'], title='Zebrahub', save="zebrafish_filtered.png")